# Relacionamentos, N+1 e Entregando Dados para o Pandas

---

## Contexto: o pipeline está lento!

O dashboard de vendas está demorando 8 segundos para carregar, sendo o dobro do aceitável.  
Você investiga e descobre: um loop está disparando centenas de queries desnecessárias.  
Esse é o famoso **problema N+1** e você vai resolvê-lo agora.

Nesta aula você vai:
- Entender lazy vs eager loading na prática
- Identificar e corrigir o problema N+1
- Usar `joinedload` e `selectinload`
- Entregar resultados em DataFrames do Pandas

---

## 1. Configuração

In [ ]:
from pathlib import Path
from datetime import datetime
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

## 2. Lazy Loading o comportamento padrão

Por padrão, o SQLAlchemy usa **lazy loading**: os dados relacionados só são buscados quando você os acessa.  

Isso é conveniente para acessos pontuais, mas **desastroso em loops**.

In [ ]:
# código:

### O problema N+1 em números

Se você tem 200 pedidos e acessa `p.cliente` em loop:

```
1 query para buscar os 200 pedidos
+ 200 queries para buscar o cliente de cada pedido
= 201 queries  ← isso é o N+1!
```

Com banco local, cada query leva ~1ms. Com PostgreSQL em produção, pode levar 10-50ms.  
**200 × 50ms = 10 segundos**  é exatamente isso que estava travando o dashboard.

## 3. Eager Loading — solução para N+1

A solução é avisar ao SQLAlchemy **antes de executar** que você vai precisar dos relacionamentos.  
Ele traz tudo de uma vez, em 1 ou 2 queries.

### 3.1 `joinedload` — usa um JOIN

In [ ]:
# código:

### 3.2 `selectinload` — usa WHERE IN

Melhor para **coleções** (um-para-muitos): 1 query principal + 1 query `WHERE id IN (...)`

In [ ]:
# código:

### Qual estratégia usar?

| Estratégia | Como funciona | Quando usar |
|---|---|---|
| `joinedload()` | 1 query com JOIN | Relacionamentos muitos-para-um (pedido → cliente) |
| `selectinload()` | 1 query + 1 `WHERE IN` | Coleções (cliente → lista de pedidos) |

---

## 4. Entregando resultados para o Pandas

No dia a dia, a maioria das análises termina em um DataFrame.  
O fluxo ideal é: **fazer o recorte no banco** (eficiente) e trazer para Python apenas o necessário.

**Padrão recomendado:**
1. `select()` com as colunas necessárias
2. `mappings()` para obter dicionários
3. `pd.DataFrame()` para montar o DataFrame

In [ ]:
# código:

In [ ]:
# código:

---

## Exercício: Usando IA para isso

**Prompt para gerar pipeline Pandas:**
```
Crie um pipeline SQLAlchemy + Pandas que:
1. Busque os pedidos do último mês com cliente e itens (sem N+1)
2. Converta para DataFrame
3. Calcule receita total por categoria de produto
4. Retorne o top 5 categorias

Modelos ORM: [cole os modelos]
```

### Resposta:Código gerado pelo Copilot


In [ ]:
from sqlalchemy import select, func
from datetime import datetime, timedelta
import pandas as pd

# Calcula o último mês
hoje = datetime.now()
ultimo_mes = hoje.replace(day=1) - timedelta(days=1)
inicio_ultimo_mes = ultimo_mes.replace(day=1)
fim_ultimo_mes = ultimo_mes

with Session(engine) as session:
    # Query para buscar pedidos do último mês com cliente e itens (JOINs explícitos evitam N+1)
    stmt = (
        select(
            Pedido.id.label("pedido_id"),
            Pedido.data_pedido,
            Pedido.valor_total,
            Cliente.nome.label("cliente_nome"),
            Cliente.estado,
            ItemPedido.quantidade,
            ItemPedido.preco_venda,
            Produto.nome.label("produto_nome"),
            Produto.categoria,
        )
        .join(Pedido.cliente)  # JOIN com Cliente
        .join(Pedido.itens)    # JOIN com ItemPedido
        .join(ItemPedido.produto)  # JOIN com Produto
        .where(Pedido.data_pedido >= inicio_ultimo_mes)  # Filtra pedidos do último mês
        .where(Pedido.data_pedido <= fim_ultimo_mes)
        # Removidas .options() pois a query é expression-based
    )
    
    dados = session.execute(stmt).mappings().all()

# Converte para DataFrame
df = pd.DataFrame(dados)

# Calcula receita total por categoria (soma de quantidade * preco_venda por categoria)
df['receita_item'] = df['quantidade'] * df['preco_venda']
receita_por_categoria = df.groupby('categoria')['receita_item'].sum().reset_index()

# Retorna top 5 categorias por receita total
top_5_categorias = receita_por_categoria.sort_values('receita_item', ascending=False).head(5)

print("Top 5 categorias por receita total no último mês:")
top_5_categorias